# AeroPulse Source Likelihood — 09: Packaging, Registry & Inference

An immutable versioned bundle, the explainable likelihood contract, and the
tests that prove it abstains instead of guessing.

## Objective

Improvement plan sections 33-40, 45-48.

- **sections 39-40** — a registry with model/feature/label/calibration
  schemas versioned together
- **section 34** — the response contract: independent likelihoods, primary
  and secondary sources, attribution confidence, unknown probability
- **section 35** — supporting evidence per source, in words
- **section 33** — abstention: the model may return "insufficient evidence"
- **section 38** — missing evidence reported as `unavailable`, never
  silently converted into a low likelihood
- **section 45** — graceful degradation when a feed is down
- **section 47** — the output statement is a safety control. Never "PM2.5 is
  caused by crop burning"; always "biomass burning is the most likely
  contributing source based on available evidence"
- **section 48** — promotion gates, which this model does not pass


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
import joblib
cal_stage = cfg.stage_dir("calibration")
clf_stage = cfg.stage_dir("classifiers")
eval_stage = cfg.stage_dir("evaluation")
gold_stage = cfg.stage_dir("gold_set")

cal_report = json.loads((cal_stage / "calibration_report.json").read_text())
clf_report = json.loads((clf_stage / "classifier_report.json").read_text())
eval_report = json.loads((eval_stage / "evaluation_report.json").read_text())
gold_report = json.loads((gold_stage / "gold_set_report.json").read_text())
base_manifest = json.loads((cfg.base_dir / "dataset_manifest.json").read_text())
label_manifest = json.loads((cfg.label_dir / "labeling_manifest.json").read_text())

calibrated = joblib.load(cal_stage / "calibrated_classifiers.joblib")
models = calibrated["models"]
FEATURES = calibrated["features"]
print("sources:", list(models), "| features:", len(FEATURES))

sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport'] | features: 56


## 1. Register an immutable bundle

In [3]:
df = pd.read_parquet(cfg.feature_dir / "source_evidence_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
labels = pd.read_parquet(cfg.label_dir / "weak_labels.parquet")
work = df.join(labels).sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

MODEL_VERSION = f"source-likelihood-{pd.Timestamp.now(tz='UTC'):%Y.%m.%d}"
registry = st.SourceRegistry(cfg.registry_dir)

evidence_reference = {
    column: float(work[column].quantile(0.75))
    for column in ("no2_level_anomaly", "so2_level_anomaly", "cams_dust",
                   "upwind_pm25_mean", "wind_speed_10m", "coarse_fraction_anomaly")
    if column in work.columns
}

bundle = {
    "classifiers": models,
    "features": FEATURES,
    "evidence_reference": evidence_reference,
    "sources": list(models),
}
test_metrics = {r["source"]: r for r in cal_report["test_metrics"]}
metadata = {
    "model_version": MODEL_VERSION,
    "model_type": f"per_source_{clf_report['winner_family']}_calibrated",
    "feature_schema_version": st.SCHEMA["schema_version"],
    "label_schema_version": st.SCHEMA["label_schema_version"],
    "sources": list(models),
    "n_features": len(FEATURES),
    "training_dataset_fingerprint": base_manifest["fingerprint"],
    "training_window": base_manifest["window"],
    "labeling_functions": label_manifest["n_functions"],
    "calibration_version": "1.0",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "git_commit": st.git_commit(),
    "gold_set_labelled": gold_report["gold_set_labelled"],
    "promoted": False,          # set by the gate evaluation below
    "caveat": "weak-supervision likelihoods; not proven source attribution",
}
metrics_payload = {
    "test": cal_report["test_metrics"],
    "temporal_spatial": eval_report["degradation"],
    "event_agreement": eval_report["event_agreement"],
    "gold_set": gold_report["gold_set_status"],
}
path = registry.register(
    bundle, metadata, metrics_payload,
    {"features": FEATURES, "excluded_label_inputs": list(st.LF_INPUT_COLUMNS)},
    {"sources": list(st.SOURCES), "schema": st.SCHEMA["label_schema_version"],
     "unknown": "derived, not a class"},
    {"methods": cal_report["selected_method"], "thresholds": cal_report["thresholds"]},
)
print("registered:", path.name)
print(registry.index().to_string(index=False))

try:
    registry.register(bundle, metadata, {}, {}, {}, {}, version=path.name)
    print("FAIL — registry overwrote a version")
except FileExistsError as exc:
    print("\nPASS — registry refused to overwrite:", str(exc)[-70:])

registered: v1
version                model_version                     model_type  promoted                       created_at
     v1 source-likelihood-2026.09.10 per_source_lightgbm_calibrated     False 2026-09-10T15:47:25.908529+00:00

PASS — registry refused to overwrite: ts/source/registry/v1 already exists — registry versions are immutable


## 2. The inference contract

In [4]:
predictor = st.SourceLikelihoodPredictor(registry)
test_start = work["timestamp_utc"].quantile(cfg.train_fraction + cfg.validation_fraction)
serving = work[work["timestamp_utc"] >= test_start]

row = serving[serving["pm25"] > 150].head(1)
if row.empty:
    row = serving.head(1)
response = predictor.predict(row)
print(json.dumps(response, indent=2, default=str))

{
  "location_id": "5616",
  "timestamp": "2026-06-14T01:30:00+00:00",
  "schema_version": "2.0.0",
  "label_schema_version": "2.0.0",
  "source_likelihoods": {
    "biomass_burning": 0.0136,
    "traffic": 0.6352,
    "industrial": 0.1275,
    "dust": 0.9677,
    "regional_transport": 0.0056
  },
  "evidence_status": {
    "biomass_burning": "available",
    "traffic": "available",
    "industrial": "available",
    "dust": "available",
    "regional_transport": "available"
  },
  "primary_source": "dust",
  "secondary_sources": [
    "traffic",
    "industrial"
  ],
  "attribution_confidence": 0.3325,
  "unknown_probability": 0.0161,
  "conflict_score": 0.6564,
  "abstained": false,
  "abstain_reason": null,
  "evidence_coverage": 1.0,
  "evidence_quality": "HIGH",
  "evidence": {
    "biomass_burning": [
      "high upwind fire radiative power"
    ],
    "traffic": [
      "within commuting hours"
    ],
    "industrial": [
      "pollution persists through the weekend"
    ],
    

## 3. Contract tests (sections 33, 38, 45, 47)

In [5]:
checks = []
def record(name, passed, detail):
    checks.append({"test": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name:<38} {detail}")

sample = serving.head(1)
out = predictor.predict(sample)

record("test_returns_all_sources", set(out["source_likelihoods"]) == set(models),
       f"{len(out['source_likelihoods'])} sources")

values = [v for v in out["source_likelihoods"].values() if v is not None]
record("test_likelihoods_independent", abs(sum(values) - 1.0) > 1e-6,
       f"sum = {sum(values):.3f} (a softmax would be exactly 1.0)")

record("test_no_causal_language",
       "proven" not in out["statement"].lower() and "caused by" not in out["statement"].lower()
       and "identified" not in out["statement"].lower(),
       out["statement"][:80])

blind = sample.copy()
blind["evidence_fire_available"] = 0
blind_out = predictor.predict(blind)
record("test_missing_evidence_reported",
       blind_out["evidence_status"].get("biomass_burning") == "unavailable"
       if "biomass_burning" in models else True,
       "fire feed down -> biomass reported unavailable, not low")

no_upwind = sample.copy()
no_upwind["evidence_upwind_available"] = 0
upwind_out = predictor.predict(no_upwind)
record("test_missing_evidence_not_negative",
       upwind_out["evidence_status"].get("regional_transport") == "unavailable"
       if "regional_transport" in models else True,
       "upwind feed down -> transport unavailable, never a 0 likelihood")

empty = sample.copy()
for column in FEATURES:
    empty[column] = np.nan
empty["evidence_coverage"] = 0.0
empty_out = predictor.predict(empty)
record("test_abstains_on_no_evidence",
       empty_out["abstained"] and empty_out["primary_source"] is None,
       f"{empty_out['abstain_reason']}")
record("test_abstention_statement", "insufficient" in empty_out["statement"].lower(),
       empty_out["statement"][:70])

record("test_evidence_explanations", any(out["evidence"].values()) if out["evidence"] else False,
       str(list(out["evidence"].items())[:1])[:90])

record("test_unknown_probability_present",
       0.0 <= out["unknown_probability"] <= 1.0,
       f"unknown_probability = {out['unknown_probability']}")

reloaded = st.SourceRegistry(cfg.registry_dir).load()
record("test_artifact_reproducible",
       set(reloaded["bundle"]["classifiers"]) == set(models)
       and reloaded["metadata"]["model_version"] == MODEL_VERSION,
       f"reloaded {reloaded['metadata']['model_version']}")

inference_tests = pd.DataFrame(checks)
print(f"\n{int(inference_tests['passed'].sum())}/{len(inference_tests)} contract tests passed")

[PASS] test_returns_all_sources               5 sources
[PASS] test_likelihoods_independent           sum = 2.076 (a softmax would be exactly 1.0)
[PASS] test_no_causal_language                Dust is the most likely contributing source based on available evidence. Traffic


[PASS] test_missing_evidence_reported         fire feed down -> biomass reported unavailable, not low
[PASS] test_missing_evidence_not_negative     upwind feed down -> transport unavailable, never a 0 likelihood
[PASS] test_abstains_on_no_evidence           insufficient evidence coverage
[PASS] test_abstention_statement              Available evidence is insufficient to indicate a likely contributing s
[PASS] test_evidence_explanations             [('traffic', ['NO2 above this station’s normal'])]
[PASS] test_unknown_probability_present       unknown_probability = 0.0161
[PASS] test_artifact_reproducible             reloaded source-likelihood-2026.09.10

10/10 contract tests passed


## 4. Behaviour over a batch

In [6]:
batch = serving.groupby("location_id", observed=True, as_index=False).tail(1).head(60)
t0 = time.perf_counter()
responses = [predictor.predict(batch.iloc[[i]]) for i in range(len(batch))]
elapsed = (time.perf_counter() - t0) * 1000

abstained = float(np.mean([r["abstained"] for r in responses]))
primaries = pd.Series([r["primary_source"] for r in responses]).value_counts(dropna=False)
confidences = pd.Series([r["attribution_confidence"] for r in responses])
latencies = pd.Series([r["latency_ms"] for r in responses])

print(f"{len(batch)} stations in {elapsed:.0f} ms ({elapsed / len(batch):.1f} ms each)")
print(f"abstention rate: {abstained:.1%}")
print("\nprimary source served:")
print(primaries.to_string())
print(f"\nattribution confidence: mean {confidences.mean():.3f}  "
      f"p10 {confidences.quantile(0.1):.3f}  p90 {confidences.quantile(0.9):.3f}")
print(f"latency: mean {latencies.mean():.1f} ms  max {latencies.max():.1f} ms")

60 stations in 696 ms (11.6 ms each)
abstention rate: 48.3%

primary source served:
NaN        29
dust       16
traffic    15

attribution confidence: mean 0.171  p10 0.014  p90 0.395
latency: mean 11.4 ms  max 12.7 ms


## 5. Promotion gates (section 48)

In [7]:
degradation = pd.DataFrame(eval_report["degradation"])
test_rows = pd.DataFrame(cal_report["test_metrics"])
geo = pd.DataFrame(eval_report["geographic"])

evidence = {
    "sources_meeting_lift": int((test_rows["pr_auc_lift"] >= st.PROMOTION_GATES["min_pr_auc_lift"]).sum()),
    "sources_below_lift": int((test_rows["pr_auc_lift"] < st.PROMOTION_GATES["min_pr_auc_lift"]).sum()),
    "worst_temporal_pr_auc_lift": float(test_rows["pr_auc_lift"].min()),
    "worst_spatial_pr_auc_lift": float(geo["pr_auc_lift"].dropna().min()) if geo["pr_auc_lift"].notna().any() else None,
    "gold_set_labelled": gold_report["gold_set_labelled"],
    "worst_ece": float(test_rows["ece"].max()),
    "unknown_rate": float(abstained),
    "missing_evidence_tested": bool(
        inference_tests.loc[inference_tests["test"] == "test_missing_evidence_reported", "passed"].iloc[0]),
    "latency_ms": float(latencies.max()),
    "artifact_reproducible": bool(
        inference_tests.loc[inference_tests["test"] == "test_artifact_reproducible", "passed"].iloc[0]),
}
decision = st.evaluate_promotion(evidence)
print("PROMOTE:" if decision["promote"] else "WITHHOLD:", decision["failed_gates"] or "-")
print()
for check in decision["checks"]:
    mark = "pass" if check["passed"] else "FAIL"
    value = check["value"]
    shown = f"{value:.4f}" if isinstance(value, float) else str(value)
    print(f"  [{mark}] {check['gate']:<26} {shown:>10}   ({check['requirement']})")

WITHHOLD: ['temporal_validation', 'spatial_validation', 'gold_set', 'no_source_degradation']

  [pass] weak_label_reproduction        4.0000   (>= 4 sources with PR-AUC lift >= 2.0)
  [FAIL] temporal_validation            1.9500   (worst-source temporal PR-AUC lift >= 2.0)
  [FAIL] spatial_validation             1.4450   (worst geographic-block PR-AUC lift >= 1.5)
  [FAIL] gold_set                       0.0000   (a human-reviewed gold set exists and has been scored)
  [pass] calibration                    0.0363   (worst-source ECE <= 0.1)
  [FAIL] no_source_degradation          1.0000   (no source below the PR-AUC lift floor)
  [pass] abstention_behaviour           0.4833   (unknown rate within [0.01, 0.9])
  [pass] missing_evidence               1.0000   (missing-evidence behaviour exercised by a test)
  [pass] latency                       12.6660   (<= 200.0 ms per request)
  [pass] reproducible_artifact          1.0000   (bundle reloads and reproduces its recorded metrics)


## 6. Drift references and deployment manifest (section 46)

In [8]:
MONITORED = ["pm25", "cams_no2", "cams_so2", "cams_dust", "no2_level_anomaly",
             "coarse_fraction_anomaly", "upwind_pm25_mean", "wind_speed_10m",
             "transport_direction_consistency", "evidence_coverage"]
rows = []
for column in MONITORED:
    if column not in work.columns:
        continue
    values = pd.to_numeric(work[column], errors="coerce").dropna()
    rows.append({"feature": column, "mean": round(float(values.mean()), 3),
                 "p05": round(float(values.quantile(0.05)), 3),
                 "p50": round(float(values.quantile(0.50)), 3),
                 "p95": round(float(values.quantile(0.95)), 3),
                 "null_pct": round(float(work[column].isna().mean() * 100), 2)})
drift_inputs = pd.DataFrame(rows)
print(drift_inputs.to_string(index=False))

drift_outputs = pd.DataFrame([{
    "source": source,
    "served_mean_likelihood": round(float(np.mean([
        r["source_likelihoods"][source] for r in responses
        if r["source_likelihoods"].get(source) is not None])), 4),
    "weak_label_rate": round(float(work[f"y_{source}"].mean()), 4),
} for source in models])
print("\nprediction reference:")
print(drift_outputs.to_string(index=False))

                        feature   mean   p05    p50     p95  null_pct
                           pm25 36.559 6.450 28.300  91.500      0.00
                       cams_no2 13.110 1.200  9.200  38.500      0.00
                       cams_so2 13.351 2.100  9.400  38.100      0.00
                      cams_dust 68.382 0.000 12.000 261.000      0.00
              no2_level_anomaly  1.150 0.152  0.917   3.014      0.00
        coarse_fraction_anomaly  3.042 0.281  1.083   8.270      0.00
               upwind_pm25_mean 36.321 8.665 27.792  88.400     36.29
                 wind_speed_10m  8.998 1.900  8.300  18.400      0.00
transport_direction_consistency  0.876 0.426  0.964   0.998      0.01
              evidence_coverage  0.927 0.800  1.000   1.000      0.00

prediction reference:
            source  served_mean_likelihood  weak_label_rate
   biomass_burning                  0.0134           0.0642
           traffic                  0.3178           0.3864
        industrial         

In [9]:
stage = cfg.stage_dir("packaging")
inference_tests.to_csv(stage / "contract_tests.csv", index=False)
drift_inputs.to_csv(stage / "drift_reference_inputs.csv", index=False)
drift_outputs.to_csv(stage / "drift_reference_outputs.csv", index=False)
(stage / "example_response.json").write_text(json.dumps(response, indent=2, default=str))

deployment = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "model_version": MODEL_VERSION,
    "registry_version": path.name,
    "git_commit": st.git_commit(),
    "promoted": decision["promote"],
    "promotion": decision,
    "contract_tests": inference_tests.to_dict(orient="records"),
    "batch_behaviour": {
        "abstention_rate": abstained,
        "mean_confidence": float(confidences.mean()),
        "mean_latency_ms": float(latencies.mean()),
    },
    "fallback_policy": [
        "a source whose evidence family is unavailable is reported as "
        "evidence_status=unavailable, never as a low likelihood",
        "coverage below the floor abstains for the whole request",
        "no source above its threshold abstains rather than forcing a primary",
        "the statement never asserts causation",
    ],
    "drift_reference": {"inputs": drift_inputs.to_dict(orient="records"),
                        "outputs": drift_outputs.to_dict(orient="records")},
    "caveat": gold_report["caveat"],
}
(stage / "deployment_manifest.json").write_text(json.dumps(deployment, indent=2, default=str))
print("09 COMPLETE ->", stage)
print("registry:", [p.name for p in sorted(cfg.registry_dir.glob("v*"))])
print("promoted:", decision["promote"])

09 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/artifacts/source/packaging
registry: ['v1']
promoted: False


## Findings

**Registered as v1, promoted: false.** Four of ten gates fail:

| gate | value | required |
|---|---|---|
| weak_label_reproduction | 4 sources | >= 4 with lift >= 2.0 — **pass** |
| temporal_validation | 1.95x | >= 2.0x — fail |
| spatial_validation | 1.45x | >= 1.5x — fail |
| gold_set | absent | must exist — fail |
| no_source_degradation | 1 source below floor | 0 — fail |
| calibration | ECE 0.036 | <= 0.10 — pass |
| abstention_behaviour | 48.3% | within [1%, 90%] — pass |
| missing_evidence | tested | pass |
| latency | 12.7 ms | <= 200 ms — pass |
| reproducible_artifact | reloaded | pass |

Three of the four failures are dust (temporal 1.95x, spatial 1.45x, and the
single source below the lift floor) and the fourth is the missing gold set.
Dust is the *most* predictable source in absolute terms (PR-AUC 0.71) and
the least impressive in lift terms, because its prevalence is 36% — there is
not much headroom above a base rate that high. The gate is doing what it
should: a 1.9x lift on a coin-flip-frequency label is not attribution.

**10/10 contract tests pass, and the ones that matter are the negative
cases.** Likelihoods sum to 2.076 rather than 1.0, confirming they are
independent rather than a disguised softmax. Taking down the fire feed makes
biomass report `evidence_status: unavailable` rather than a low likelihood —
the plan's section 45 rule that missing evidence must never become negative
evidence. Stripping every feature produces an explicit abstention with the
reason `insufficient evidence coverage`, not a guess. And the statement test
rejects "proven", "caused by" and "identified", which is a safety control
rather than a style preference (section 47).

**The served behaviour is dominated by abstention: 48.3% of 60 stations get
no primary source at all.** For a source-attribution product this is the
right failure mode and an uncomfortable headline. Of those that do resolve,
16 are dust and 15 traffic — biomass, industrial and regional transport are
served to nobody, because their calibrated likelihoods sit below thresholds
tuned on a period where they were far more common.

**Mean attribution confidence is 0.171** (p90 0.395). The model is telling
the truth: it is not confident, and the contract surfaces that rather than
hiding it behind a normalised top-1 label.

**Prediction drift against the label rate is visible already** and worth
carrying into monitoring: served mean likelihood versus weak-label rate is
0.013 vs 0.064 for biomass and 0.025 vs 0.132 for industrial — the served
probabilities are roughly 5x below their label prevalence, a direct
consequence of the train/test prevalence collapse in notebook 06.

**Latency is not a constraint**: 11.6 ms per station for all five sources.

What production would serve today is an abstention for half of requests and
a low-confidence dust or traffic likelihood for the rest. That is the system
behaving correctly, and it is the honest summary of a source-attribution
model that has never been compared against a human-reviewed label.